# LSTM training

Trains one `TrainConfig` on the rolling turn windows, logs every epoch to MLflow, and saves the best-validation checkpoint to the volume.

Attach to **Serverless GPU, A10**. The data is `data/lstm/episodes.jsonl` copied to the volume; the code comes from this Git folder, whose root Databricks puts on `sys.path`.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import mlflow
import numpy as np
import torch

from drzavni_posao.generation.lstm.prepare import read_jsonl
from drzavni_posao.generation.lstm.train import TrainConfig, save_checkpoint, train
from drzavni_posao.transcription.constants import CORE_CHARACTERS

VOLUME = Path("/Volumes/luka_erceg/drzavni_posao/lstm")
EPISODES_PATH = VOLUME / "episodes.jsonl"
CHECKPOINT_DIR = VOLUME / "checkpoints"
EXPERIMENT_NAME = "/Users/luka.erceg@databricks.com/drzavni_posao/lstm"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.__version__, torch.cuda.get_device_name() if DEVICE == "cuda" else "")

In [ ]:
RUN_NAME = "lstm-k5"
CONFIG = TrainConfig(
    cell="lstm",
    hidden_size=512,
    num_layers=3,
    dropout=0.2,
    context_turns=5,
    premise_mode="none",
    bpe_vocab_size=None,
)
SAMPLES_PER_SPEAKER = 3
TEMPERATURE = 0.8
CONFIG

In [ ]:
episodes = read_jsonl(EPISODES_PATH)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_experiment(EXPERIMENT_NAME)


def log_epoch(epoch: int, metrics: dict[str, float]) -> None:
    mlflow.log_metrics(metrics, step=epoch)
    print(f"epoch {epoch}: " + ", ".join(f"{k} {v:.4g}" for k, v in metrics.items()))


with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params(asdict(CONFIG))
    model, results = train(CONFIG, episodes, DEVICE, on_epoch=log_epoch)
    mlflow.log_metrics({f"best_{k}": v for k, v in results.items()})

    checkpoint = CHECKPOINT_DIR / f"{RUN_NAME}-{run.info.run_id}.pt"
    save_checkpoint(model, CONFIG, checkpoint)
    mlflow.log_artifact(str(checkpoint))

    rng = np.random.default_rng(CONFIG.seed)
    newline = model.vocab.ids["\n"]
    samples = []
    for speaker in CORE_CHARACTERS:
        tag = f"[{speaker}]"
        for _ in range(SAMPLES_PER_SPEAKER):
            ids = model.generate([model.vocab.ids[tag]], newline, rng, TEMPERATURE)
            samples.append(f"{tag} {model.vocab.decode(ids).strip()}")
    mlflow.log_text("\n".join(samples), "samples.txt")

print(results)
print("\n".join(samples))